# Dutch North Sea Decommissioning — Exploratory Notebook

Seven sections, built and committed one at a time.

| Section | Purpose |
|---|---|
| 1 | Fetch raw borehole data from NLOG API |
| 2 | Inspect shape, dtypes, uniques, nulls |
| 3 | Filter offshore, split Group A / B / ghost |
| 4 | Operator analysis on Group A |
| 5 | Production crossref — informal cessation |
| 6 | Join and summarise |
| 7 | Sense check against data diary |

**Data dirs** (`netherlands/data/`) are gitignored. Raw JSON and all CSVs  
live only on your local machine unless explicitly exported.

---
## Section 1 — Fetch

Uses `requests.Session` to:
1. `GET` the datacenter overview page — sets the required session cookie.
2. `POST` (empty body) to the boreholes endpoint — returns all 6,723 wells.

Saves raw JSON to `netherlands/data/raw/nlog_boreholes_raw.json`.

In [ ]:
import json
import logging
from pathlib import Path

import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Paths — notebook lives in netherlands/notebooks/, data two levels over.
# Path.cwd() resolves to the notebook directory when launched normally.
# ---------------------------------------------------------------------------
NOTEBOOKS_DIR = Path.cwd()
NETHERLANDS_ROOT = NOTEBOOKS_DIR.parent          # netherlands/
DATA_RAW = NETHERLANDS_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

RAW_JSON_PATH = DATA_RAW / "nlog_boreholes_raw.json"

print(f"Saving to: {RAW_JSON_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_SEED_URL = "https://www.nlog.nl/datacenter/brh-overview"
_BOREHOLES_URL = "https://www.nlog.nl/nlog-mapviewer/rest/brh/boreholes"
_TIMEOUT = 60  # seconds
_EXPECTED_COUNT = 6_723
_EXPECTED_FIELDS = {
    "boreholeDbk",
    "boreholeName",
    "shortName",
    "clientOrgName",
    "legalOwnerName",
    "statusDescription",
    "resultCode",
    "onOffshore",
    "startDate",
    "endDate",
    "confidentialityDate",
    "blockCd",
}

In [ ]:
def seed_session() -> requests.Session:
    """Open a session and hit the datacenter overview page to set the required cookie."""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; research/1.0)"})
    resp = session.get(_SEED_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    logger.info("Seed GET %s → %s (cookies: %s)", _SEED_URL, resp.status_code, list(session.cookies.keys()))
    return session


def fetch_boreholes(session: requests.Session) -> list:
    """POST to the boreholes endpoint and return the parsed JSON array."""
    resp = session.post(_BOREHOLES_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    logger.info("Boreholes POST → %s records", len(data))
    return data


def validate_record_count(data: list) -> None:
    """Warn loudly if the record count differs significantly from the expected total."""
    count = len(data)
    delta = abs(count - _EXPECTED_COUNT)
    if delta > 50:
        logger.warning(
            "COUNT MISMATCH — got %d records, expected ~%d (delta %d). "
            "Stop and flag before proceeding.",
            count, _EXPECTED_COUNT, delta,
        )
    else:
        logger.info("Record count %d — within 50 of expected %d ✓", count, _EXPECTED_COUNT)


def validate_field_names(record: dict) -> None:
    """Check that every expected field is present in the first record."""
    actual = set(record.keys())
    missing = _EXPECTED_FIELDS - actual
    extra = actual - _EXPECTED_FIELDS
    if missing:
        logger.warning("MISSING FIELDS: %s", missing)
    if extra:
        logger.info("Extra fields not in spec (note for data diary): %s", extra)
    if not missing:
        logger.info("All expected fields present ✓")


def save_raw(data: list, path: Path) -> None:
    """Write the raw JSON array to disk."""
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    logger.info("Saved %d records to %s", len(data), path)

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 1
# ---------------------------------------------------------------------------

session = seed_session()
raw_data = fetch_boreholes(session)

validate_record_count(raw_data)
validate_field_names(raw_data[0])

print("\n--- First record ---")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

save_raw(raw_data, RAW_JSON_PATH)